# TAREA 6
María Fernanda Gómez Flores  
Carlos Hassan Hernández Rohan  
Portafolios de Inversión

## PUNTO A

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
from scipy.optimize import minimize

### 1. MÍNIMA SEMIVARIANZA

In [ ]:
# Descarga de Datos
prices = yf.download(['PG', 'COST', 'KO', 'WMT', 'CLX', 'KHC'],
                     start='2016-01-01', end='2026-10-15')['Close']

prices.isna().sum()

In [ ]:
# Calculo de rendimientos
rets=prices.pct_change().dropna()

In [ ]:
# Auditoría de fechas
print(prices.index.min(), prices.index.max())
print(prices.isna().sum())
print(rets.index.min(), rets.index.max())
print(len(rets))

In [ ]:
def opt_min_semivar(prices: pd.DataFrame):
    #Calcular los rendimientos
    rets = prices.pct_change().dropna()

    #Calcular matriz de correlación
    corr = rets.corr()

    #Calcular downside risk
    below_zero = (rets[rets<0]).fillna(0)
    downside = np.array(below_zero.std())

    #Calcular matriz de semivarianza
    n_assets = len(prices.columns)
    semivar_matrix = (downside.reshape(n_assets,1) @ downside.reshape(1,n_assets))*corr

    # Funcion.
    semivar = lambda w: w.T @ semivar_matrix @ w

    #Variables iniciales (pesos)
    w_0 = np.ones(n_assets) / n_assets

    # Restricciones.
    restr = lambda w: sum(w) - 1

    #rangos de las variables de decisión
    rangos = [(0,1)] * n_assets

    # tolerancia
    tol = 1e-32

    #Optimizar
    results = minimize(fun=semivar, x0=w_0, bounds=rangos,
                       constraints={'fun': restr, 'type': 'eq'}, tol=tol)

    return results.x

In [ ]:
# Verificar resultados de la función
w_min_semivar=opt_min_semivar(prices)
dict(zip(prices.columns,w_min_semivar))

#### Comprobación Montecarlo

In [ ]:
below_zero=(rets[rets<0]).fillna(0)
below_zero

In [ ]:
downside=np.array(below_zero.std())
downside

In [ ]:
n_assets=len(prices.columns)
#downside @ downside.T
semivar_matrix=(downside.reshape(n_assets,1) @ downside.reshape(1,n_assets))*rets.corr()
semivar_matrix

In [ ]:
semivar_port=lambda w: w.T @ semivar_matrix @ w

In [ ]:
# Obtener semivarianza mínima
semivar_port(w_min_semivar)

In [ ]:
# Simular portafolios con montecarlo
np.random.seed(42)
n_sim=10000
w_random=np.random.dirichlet(np.ones(n_assets), size=n_sim)

In [ ]:
# Obtener la mínima semivarianza
semivars_port=[semivar_port(w_i) for w_i in w_random]
np.min(semivars_port)

In [ ]:
# Obtener los pesos del portafolio con mínima semivarianza
w_semivar=w_random[np.argmin(semivars_port)]
dict(zip(prices.columns, w_semivar))

In [ ]:
# Comparar resultados
comparacion=pd.DataFrame({
    'SciPy':w_min_semivar*100,
    'Montecarlo':w_semivar*100
},index=prices.columns)

comparacion

In [ ]:
# Comparar semivarianza mínima
resultados_semivar=pd.DataFrame({
    'Semivarianza':[semivar_port(w_min_semivar),np.min(semivars_port)]
},index=['SciPy','Montecarlo'])
resultados_semivar

**Conclusión**  
En mínima semivarianza obtuvimos 0.0000354124 con SciPy y 0.0000356383 con Montecarlo, entonces los dos métodos quedaron bastante cercanos. En SciPy la mayor ponderación fue KO con 33.43%, seguido de PG con 21.10% y WMT con 19.74%. En Montecarlo también quedó KO con el mayor peso, 34.35%. Entonces sí se ve que los dos métodos llegan a una distribución parecida, aunque SciPy encontró una semivarianza un poco menor.

### 2. MÁXIMO OMEGA

In [ ]:
def opt_max_omega(prices: pd.DataFrame):
    #Calcular los rendimientos
    rets = prices.pct_change().dropna()

    #Calcular downside y upside risk
    below_zero=rets[rets<0].fillna(0)
    above_zero=rets[rets>0].fillna(0)
    upside_risk=np.array(above_zero.std())
    downside_risk=np.array(below_zero.std())

    # Calcular Omega individual
    omega=upside_risk/downside_risk

    # Funcion
    func_omega=lambda w: -sum(w*omega)

    # Restricciones
    restr=lambda w: sum(w)-1

    # Valor Inicial
    n_assets=len(prices.columns)
    w_0=np.ones(n_assets)/n_assets

    # tol
    tol=1e-32

    # Rangos
    rangos=[(0,1)]*n_assets

    # optimización
    results=minimize(fun=func_omega,x0=w_0,bounds=rangos,
                     constraints={'fun':restr,'type':'eq'},tol=tol)

    return results.x

In [ ]:
# Obtener los pesos
w_max_omega=opt_max_omega(prices)
dict(zip(prices.columns,w_max_omega))

#### Comprobación Montecarlo

In [ ]:
below_zero=rets[rets<0].fillna(0)
above_zero=rets[rets>0].fillna(0)

In [ ]:
upside_risk=np.array(above_zero.std())
upside_risk

In [ ]:
downside_risk=np.array(below_zero.std())
downside_risk

In [ ]:
omega=upside_risk/downside_risk
omega

In [ ]:
# Obtener el maximo omega
func_omega=lambda w: sum(w*omega)

In [ ]:
# Simular portafolios con montecarlo
w_random=np.random.dirichlet(np.ones(n_assets), size=n_sim)

In [ ]:
# Obtener el maximo omega
omegas_port=[func_omega(w_i) for w_i in w_random]
np.max(omegas_port)

In [ ]:
# Obtener los pesos del portafolio con maximo omega
w_omega_mc=w_random[np.argmax(omegas_port)]
dict(zip(rets.columns,w_omega_mc))

In [ ]:
# Comparar resultados
comparacion_omega=pd.DataFrame({
    'SciPy':w_max_omega*100,
    'Montecarlo':w_omega_mc*100
},index=prices.columns)
comparacion_omega

In [ ]:
# Comparar Omega máximo
resultados_omega=pd.DataFrame({
    'Omega':[func_omega(w_max_omega),np.max(omegas_port)]
},index=['SciPy','Montecarlo'])
resultados_omega

**Conclusión**  
En máximo Omega cambió muchísimo la distribución. Con SciPy todo el peso quedó en WMT con 100%, mientras que Montecarlo le asignó 90.36%. El Omega de SciPy fue 1.0704 y el de Montecarlo 1.0575, entonces también quedaron bastante cercanos. Esto pasó porque WMT tuvo el Omega individual más alto. Estamos usando el Omega ponderado que vimos en clase, no el Omega calculado directamente sobre los rendimientos del portafolio. Una desventaja es que el resultado quedó concentrado en una sola empresa.

### 3. SEMIVARIANZA OBJETIVO

In [ ]:
# Descarga de Precios del Benchmark
benchmark = yf.download('KXI', start='2016-01-01', end='2026-10-15')['Close']
benchmark

In [ ]:
# Obtener rendimientos del benchmark
rets_benchmark=benchmark.pct_change().dropna()
rets_benchmark

In [ ]:
# Auditoría de fechas
print(prices.index.min(), prices.index.max())
print(benchmark.index.min(), benchmark.index.max())
print(prices.isna().sum())
print(benchmark.isna().sum())

In [ ]:
# Comprobar fechas en común
fechas=rets.index.intersection(rets_benchmark.index)
print(len(rets),len(rets_benchmark),len(fechas))
print(fechas.min(),fechas.max())
print(rets.loc[fechas].isna().sum())
print(rets_benchmark.loc[fechas].isna().sum())

In [ ]:
def opt_target_semivar(prices: pd.DataFrame, benchmark: pd.Series):
    #Calcular los rendimientos
    rets=prices.pct_change().dropna()
    rets_benchmark=benchmark.pct_change().dropna()

    # Sustraer rendimiento del benchmark de rendimientos individuales
    alpha=rets.sub(rets_benchmark.squeeze(),axis=0).dropna()

    # Separar los rendimientos por debajo de cero
    below_zero_t=(alpha[alpha<0]).fillna(0)

    # Calcular target downside risk
    downside_risk_t=np.array(below_zero_t.std())

    # Calcular la matriz de target semivariance
    corr=alpha.corr()
    n_assets=len(prices.columns)
    semivar_matrix=(downside_risk_t.reshape(n_assets,1) @ downside_risk_t.reshape(1,n_assets))*corr

    # Funcion
    func_target=lambda w: w.T @ semivar_matrix @ w

    # Restricciones
    restr=lambda w: sum(w)-1

    # Valor Inicial
    w_0=np.ones(n_assets)/n_assets

    # tol
    tol=1e-32

    # Rangos
    rangos=[(0,1)]*n_assets

    # optimización
    results=minimize(fun=func_target,x0=w_0,bounds=rangos,
                     constraints={'fun':restr,'type':'eq'},tol=tol)

    return results.x

In [ ]:
# Obtener los pesos del portafolio con mínima semivarianza target
w_target_semivar=opt_target_semivar(prices,benchmark)
dict(zip(prices.columns,w_target_semivar))

#### Comprobación Montecarlo

In [ ]:
# Sustraer rendimiento del benchmark de rendimientos individuales
alpha=rets.sub(rets_benchmark.squeeze(),axis=0).dropna()
alpha

In [ ]:
# Separar los rendimientos por debajo de cero
below_zero_t=(alpha[alpha<0]).fillna(0)
below_zero_t

In [ ]:
# Calcular target downside risk
downside_risk_t=np.array(below_zero_t.std())
downside_risk_t

In [ ]:
# Calcular la matriz de target semivariance
n_assets=len(prices.columns)
#downside @ downside.T
semivar_matrix_t=(downside_risk_t.reshape(n_assets,1) @ downside_risk_t.reshape(1,n_assets))*alpha.corr()
semivar_matrix_t

In [ ]:
# Simular portafolios con montecarlo
w_random=np.random.dirichlet(np.ones(n_assets), size=n_sim)

In [ ]:
# Obtener la mínima target semivariance
func_target = lambda w: w.T @ semivar_matrix_t @ w
mts=[func_target(w_i)for w_i in w_random]
np.min(mts)

In [ ]:
# Obtener los pesos del portafolio con mínima semivarianza target
w_tarjet_semivar = w_random[np.argmin(mts)]
dict(zip(rets.columns, w_tarjet_semivar))

In [ ]:
# Comparar resultados
comparacion_target=pd.DataFrame({
    'SciPy':w_target_semivar*100,
    'Montecarlo':w_tarjet_semivar*100
},index=prices.columns)

comparacion_target

In [ ]:
# Comparar semivarianza objetivo
resultados_target=pd.DataFrame({
    'Semivarianza objetivo':[func_target(w_target_semivar),np.min(mts)]
},index=['SciPy','Montecarlo'])
resultados_target.apply(lambda x: x.map(lambda v: f'{v:.10f}'))

**Conclusión**  
En semivarianza objetivo usamos KXI como benchmark, entonces buscamos disminuir el riesgo de obtener rendimientos por debajo de ese ETF. Con SciPy la mayor ponderación fue KO con 33.54%, seguido de PG con 27.54% y COST con 15.76%. En Montecarlo también quedaron KO con 35.20% y PG con 25.39%. El resultado de Montecarlo fue 0.0000078766. Para comparar bien cuál método obtuvo la menor semivarianza objetivo, dejamos los valores con 10 decimales en la tabla anterior, porque con 6 decimales los dos se veían iguales.